# Experiment E2: ERA5 forcing, the new baseline. RUN THIS FIRST (Hurricane Ike 2008)

**Changes from E0 (`gom12_ike.001`):** the **wind (u10, v10) and sea-level pressure (msl)** streams come from **ERA5** (0.25 deg, hourly) instead of JRA55-do (~55 km, 3-hourly). Everything else in the atmosphere (temperature, humidity, radiation, rain) stays JRA.
**Same as E0:** 1/12 deg ocean grid, bathymetry, GLORYS, TPXO, dates.

Because the ocean grid is unchanged, E2 is a **clone of E0 with the compiled model reused** (`--keepexe`): no new grid, forcing processing or build.

**Run this first.** E2 is the new baseline: E1 and E3 reuse the ERA5 files made here (sections 2 and 3) and compare against E2.

**Question:** do better winds recover the forerunner and the peak? This is expected to be the largest effect.
**Compare with:** E0 (section 7d).

**Known limit:** DATM remaps all streams onto its own atmosphere grid (TL319, ~0.5 deg), which smooths ERA5 somewhat. Ask the instructors whether a finer atmosphere grid is possible.

**Where to run:** Casper JupyterHub, kernel **CrocoDash**, 40 GB memory (the global ERA5 files are a few GB). Plan: `EXPERIMENT_PLAN.md`.

**Status:** written, not yet run. The ERA5 path on GLADE (NCAR RDA ds633.0) and the stream swap are unverified: check the printouts at each step.


## 0. Settings

In [ ]:
import os
from pathlib import Path

WORKSPACE = Path("/glade/work/faezehmaghso/crocodile2026")
CESMROOT  = WORKSPACE / "CESM"
BASE_CASE = "gom12_ike.001"                  # finished E0 run (1/12 deg, JRA)
CASENAME  = "gom12_ike_E2.001"               # clone of BASE_CASE with ERA5 wind + pressure
CASEROOT  = WORKSPACE / "croc_cases" / CASENAME
INPUTDIR  = WORKSPACE / "croc_input" / BASE_CASE   # shared with E0 (read only)

# ERA5 surface analysis on GLADE (NCAR RDA ds633.0); monthly files per variable
ERA5_DIR  = Path("/glade/campaign/collections/rda/data/d633000/e5.oper.an.sfc")
ERA5_CODES = {"u10": "128_165_10u", "v10": "128_166_10v", "msl": "128_151_msl"}
MONTHS    = ["200808", "200809"]
T0, T1    = "2008-08-29T00", "2008-09-21T00"  # covers the run (08-30 -> 09-20) with a day to spare

FORCE_DIR = WORKSPACE / "era5_forcing" / "ike2008"
MESH      = FORCE_DIR / "era5_0.25deg_global_ESMFmesh.nc"
print(CASEROOT, CASEROOT.exists())

## 1. Clone E0 (Derecho terminal)

Run one line at a time:
```bash
cd /glade/work/faezehmaghso/crocodile2026/croc_cases
/glade/work/faezehmaghso/crocodile2026/CESM/cime/scripts/create_clone --clone gom12_ike.001 --case gom12_ike_E2.001 --keepexe
cd gom12_ike_E2.001
./case.setup
./xmlquery JOB_QUEUE          # should already be tutorial; if not: ./xmlchange JOB_QUEUE=tutorial --force
```
`--keepexe` reuses E0's compiled model, so **no build** is needed. Then re-run the settings cell: `CASEROOT exists` should print `True`.

## 2. ERA5 wind and pressure files

Reads the global ERA5 hourly fields for 29 Aug to 21 Sep 2008 and writes one NetCDF per variable (time, lat, lon), lat ascending.
Global files avoid any gap when DATM remaps them onto its global atmosphere grid.

If no files are found, check the path with `ls /glade/campaign/collections/rda/data/d633000/e5.oper.an.sfc/200809/ | head`, or download u10, v10, msl for these dates from the Copernicus Climate Data Store (reanalysis-era5-single-levels) into `FORCE_DIR` with the same file names.

In [ ]:
import glob
import numpy as np
import xarray as xr

FORCE_DIR.mkdir(parents=True, exist_ok=True)
out_files = {}
for name, codeid in ERA5_CODES.items():
    out = FORCE_DIR / f"era5_{name}_ike2008.nc"
    out_files[name] = out
    if out.exists():
        print("exists:", out.name)
        continue
    files = sorted(f for m in MONTHS for f in glob.glob(str(ERA5_DIR / m / f"*{codeid}*.nc")))
    if not files:
        raise FileNotFoundError(f"No ERA5 {name} files in {ERA5_DIR}/{MONTHS}. See the note above.")
    ds = xr.open_mfdataset(files, combine="by_coords")
    var = next(v for v in ds.data_vars if ds[v].ndim == 3)
    lat = next(n for n in ["latitude", "lat"] if n in ds.dims)
    lon = next(n for n in ["longitude", "lon"] if n in ds.dims)
    da = (ds[var].sel(time=slice(T0, T1)).rename({lat: "lat", lon: "lon"})
          .sortby("lat").sortby("lon").astype("float32"))
    da.name = name
    enc = {name: {"_FillValue": None},
           "time": {"units": "hours since 2008-01-01 00:00:00", "calendar": "gregorian", "dtype": "float64"},
           "lat": {"_FillValue": None}, "lon": {"_FillValue": None}}
    da.to_dataset().to_netcdf(out, encoding=enc)
    print(f"{name}: from {var} in {len(files)} files -> {out.name}, {dict(da.sizes)}, "
          f"{str(da.time.values[0])[:13]} to {str(da.time.values[-1])[:13]}")

chk = xr.open_dataset(out_files["msl"])
print("msl range (Pa):", float(chk.msl.min()), float(chk.msl.max()), " (Ike minimum near 95000 Pa expected)")

## 3. ESMF mesh for the ERA5 grid

CDEPS needs an ESMF mesh file that describes the grid of the new streams. Element order is lat-outer, lon-inner, matching the data layout.

In [ ]:
def write_esmf_mesh(lon, lat, path):
    lon = np.asarray(lon, float)
    lat = np.asarray(lat, float)
    nx, ny = lon.size, lat.size
    dlon, dlat = np.diff(lon).mean(), np.diff(lat).mean()
    lon_e = np.concatenate([lon - dlon / 2, [lon[-1] + dlon / 2]])
    lat_e = np.clip(np.concatenate([lat - dlat / 2, [lat[-1] + dlat / 2]]), -90, 90)
    lon_n, lat_n = np.meshgrid(lon_e, lat_e)                       # (ny+1, nx+1) corner nodes
    nodes = np.stack([lon_n.ravel(), lat_n.ravel()], axis=1)
    j, i = np.meshgrid(np.arange(ny), np.arange(nx), indexing="ij")
    n0 = (j * (nx + 1) + i).ravel()
    conn = np.stack([n0, n0 + 1, n0 + nx + 2, n0 + nx + 1], axis=1) + 1   # counter-clockwise, 1-based
    lon_c, lat_c = np.meshgrid(lon, lat)
    centers = np.stack([lon_c.ravel(), lat_c.ravel()], axis=1)
    mesh = xr.Dataset(
        {"nodeCoords": (("nodeCount", "coordDim"), nodes, {"units": "degrees"}),
         "elementConn": (("elementCount", "maxNodePElement"), conn.astype("int32"),
                         {"long_name": "Node Indices that define the element connectivity", "start_index": 1}),
         "numElementConn": (("elementCount",), np.full(nx * ny, 4, dtype="int32")),
         "centerCoords": (("elementCount", "coordDim"), centers, {"units": "degrees"}),
         "elementMask": (("elementCount",), np.ones(nx * ny, dtype="int32"))},
        attrs={"gridType": "unstructured mesh"})
    mesh.to_netcdf(path, encoding={"elementConn": {"_FillValue": -1}})


if not MESH.exists():
    g0 = xr.open_dataset(out_files["u10"])
    write_esmf_mesh(g0.lon.values, g0.lat.values, MESH)
m = xr.open_dataset(MESH)
print(MESH.name, dict(m.sizes))

## 4. Point the JRA wind and pressure streams at ERA5

Adds a block to `user_nl_datm_streams`, **after** CrocoDash's year-alignment block (in CDEPS the last entry wins).
It finds the stream names ending in `U_10`, `V_10` and `SLP_` and sets `datafiles`, `meshfile`, `datavars`, the years, and `taxmode = limit` (stops loudly if the run asks for a time outside the ERA5 files).

In [ ]:
import re

TAG_B, TAG_E = "! >>> E2: ERA5 wind + pressure", "! <<< E2: ERA5 wind + pressure"
nl = CASEROOT / "user_nl_datm_streams"
txt = nl.read_text() if nl.exists() else ""
txt = re.sub(re.escape(TAG_B) + r".*?" + re.escape(TAG_E) + r"\n?", "", txt, flags=re.S)   # remove old block

names = set(re.findall(r"^\s*([A-Za-z0-9_.]+)\s*:", txt, re.M))
if not names:
    xml = CASEROOT / "CaseDocs" / "datm.streams.xml"
    if xml.exists():
        names = set(re.findall(r'name="([^"]+)"', xml.read_text()))
assert names, "No DATM stream names found: run ./preview_namelists in the case folder, then re-run this cell."

lines = [TAG_B]
for key, suffix, datavar in [("u10", "U_10", "u10 Sa_u"), ("v10", "V_10", "v10 Sa_v"), ("msl", "SLP_", "msl Sa_pslv")]:
    cand = sorted(n for n in names if n.endswith(suffix))
    assert len(cand) == 1, f"expected one stream ending in {suffix}, found {cand}"
    s = cand[0]
    lines += [f"{s}:datafiles = {out_files[key]}",
              f"{s}:meshfile = {MESH}",
              f"{s}:datavars = {datavar}",
              f"{s}:year_first = 2008",
              f"{s}:year_last = 2008",
              f"{s}:year_align = 2008",
              f"{s}:taxmode = limit"]
lines += [TAG_E, ""]
nl.write_text(txt.rstrip() + "\n\n" + "\n".join(lines))
print("\n".join(lines))

## 5. Output (diag_table, same as E0 plus wind stress)

Rewrites the clone's SFINCS output block with the E2 case name, and adds `taux`, `tauy`, `SSU`, `SSV` (Texas box, every 2 h).

In [ ]:
# Writes SourceMods/src.mom/diag_table. MOM_interface copies a resolved diag_table from
# SourceMods straight into the run folder, so ./preview_namelists is NOT needed first.
import shutil

src = CASEROOT / "CaseDocs" / "diag_table"          # CESM defaults (exists after a preview_namelists)
dst = CASEROOT / "SourceMods" / "src.mom" / "diag_table"
MARK = "### SFINCS nesting output"                  # everything from here on is ours


def box(w, e, s, n):
    # FMS regional_section: lon0 lon1 lat0 lat1 z0 z1, lon in 0-360 like the grid
    return f"{w % 360:.2f} {e % 360:.2f} {s:.2f} {n:.2f} -1 -1"


c = CASENAME
sfincs = [
    "",
    f"{MARK} (added by notebook section 5)",
    f'"{c}.mom6.sfincs_TX", 10, "minutes", 1, "days", "time"',
    f'"{c}.mom6.sfincs_MS", 10, "minutes", 1, "days", "time"',
    f'"{c}.mom6.ssh_hourly", 1, "hours", 1, "days", "time"',
    # wind stress + surface currents, Texas box, every 2 h (posted once per 1-h coupling step,
    # so a 2-h instantaneous window always holds data)
    f'"{c}.mom6.wind_TX", 2, "hours", 1, "days", "time"',
    f'"ocean_model", "SSH_inst", "SSH_inst", "{c}.mom6.sfincs_TX", "all", ".false.", "{box(-97.5, -93.0, 27.5, 30.0)}", 2',
    f'"ocean_model", "SSH_inst", "SSH_inst", "{c}.mom6.sfincs_MS", "all", ".false.", "{box(-91.0, -87.5, 28.5, 30.8)}", 2',
    # SSH_inst (posted every DT), not the SSH mean: the SSH mean is posted once per coupling
    # step and an hourly average of it can be empty -> FMS "write EMPTY buffer" FATAL.
    f'"ocean_model", "SSH_inst", "SSH_inst", "{c}.mom6.ssh_hourly", "all", ".false.", "none", 2',
] + [
    f'"ocean_model", "{v}", "{v}", "{c}.mom6.wind_TX", "all", ".false.", "{box(-97.5, -93.0, 27.5, 30.0)}", 2'
    for v in ["taux", "tauy", "SSU", "SSV"]
] + [
    "",
]

if src.exists():
    # CESM default outputs, minus any SFINCS block from an earlier run of this cell
    base = src.read_text().split(MARK)[0].rstrip() + "\n"
    how = "CESM default diag_table (CaseDocs) + SFINCS outputs"
else:
    # No CaseDocs yet: write a small, complete diag_table ourselves
    base = "\n".join(
        [f'"MOM6 diagnostic fields table for CESM case: {c}"', "1 1 1 0 0 0", "",
         f'"{c}.mom6.h.static", -1, "days", 1, "days", "time"',
         f'"{c}.mom6.h.sfc_daily", 1, "days", 1, "days", "time"']
        + [f'"ocean_model", "{v}", "{v}", "{c}.mom6.h.static", "all", "none", "none", 2'
           for v in ["areacello", "geolon", "geolat", "deptho", "wet"]]
        + [f'"ocean_model", "{v}", "{v}", "{c}.mom6.h.sfc_daily", "all", ".true.", "none", 2'
           for v in ["tos", "sos", "SSU", "SSV"]]
    ) + "\n"
    how = "minimal diag_table (no CaseDocs yet) + SFINCS outputs"

# MOM6 declares areacello as the AREA measure for deptho. Some minimal or
# previously generated CaseDocs tables omit it, which is fatal at startup.
if '"areacello"' not in base:
    base += f'"ocean_model", "areacello", "areacello", "{c}.mom6.h.static", "all", "none", "none", 2\n'
    how += " + required areacello AREA measure"

dst.parent.mkdir(parents=True, exist_ok=True)
dst.write_text(base + "\n".join(sfincs))
n = sum(1 for line in dst.read_text().splitlines() if any(k in line for k in ("sfincs_TX", "sfincs_MS", "ssh_hourly")))
print(f"wrote {dst}\n  {how}\n  SFINCS lines: {n} (must be 6)")
print("\n".join(sfincs))

## 6. Check and run (Derecho terminal)

One line at a time, in `/glade/work/faezehmaghso/crocodile2026/croc_cases/gom12_ike_E2.001`:
```bash
./preview_namelists
grep -c "era5_" $(./xmlquery --value RUNDIR)/datm.streams.xml      # must print 6 (3 data files + 3 mesh lines)
grep -B2 -A12 "NCEP.U_10" $(./xmlquery --value RUNDIR)/datm.streams.xml | head -30   # U_10 now reads era5_u10
grep -c "sfincs_TX\|sfincs_MS\|ssh_hourly" $(./xmlquery --value RUNDIR)/diag_table     # must print 6
grep -c "wind_TX" $(./xmlquery --value RUNDIR)/diag_table                               # must print 5
./case.submit
qstat -u faezehmaghso
tail -8 CaseStatus
```
No build is needed (`--keepexe`). Expected run time about 8 min, like E0.

**If the run fails**, look in the atmosphere log first (the new streams are read by DATM):
```bash
grep -n -i "error\|abort" $(./xmlquery --value RUNDIR)/atm.log.* | head
grep -n -i -B3 -A6 "FATAL" $(./xmlquery --value RUNDIR)/ocn.log.*
```
Paste the lines; stream problems (mesh, time axis, variable names) show up there.

### Submit on the correct queue (workshop `tutorial` reservation)

The **model run** and the **archiver** must use the workshop queue `tutorial`.
CrocoDash creates the case with `main`, so change it **after** `./case.setup --reset` and **before** `./case.submit`:
```bash
./xmlchange JOB_QUEUE=tutorial --force        # --force: CESM doesn't know this workshop queue
./xmlquery JOB_QUEUE,JOB_WALLCLOCK_TIME       # case.run and case.st_archive: tutorial, 02:00:00
./case.submit                                 # the qsub line must show "-q tutorial"
qstat -u faezehmaghso                         # Queue column shows S7606483 = the tutorial reservation
```
- The **build** (`qcmd -A UCGD0009 -- ./case.build`) always runs on the small `develop` queue. That is normal.
- **Already submitted on `main` by mistake?** Cancel both jobs with `qdel <run-id> <archive-id>` (IDs from `qstat -u faezehmaghso`), then run the lines above.
- `qstat -u faezehmaghso` briefly showing nothing right after submitting is normal; check again or run `tail -8 CaseStatus`.
- **After the workshop** the reservation ends: switch back with `./xmlchange JOB_QUEUE=main --force` (and expect longer waits).


## 7. After the run: join, save, check

The box outputs (`sfincs_TX`, `sfincs_MS`) are written in **pieces** (`.nc.0088`, ...), one per processor.
They stay in the **run folder on scratch**, because the archiver skips them.
The next cell joins the pieces and saves them to `/glade/work/.../mom6_output/<case>/`.

**Calendar:** MOM6 writes `calendar: gregorian`, but means proleptic Gregorian. Without the fix, dates appear **2 days early**.

Run the **settings cell (0)** first if the kernel was restarted.

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shutil
from pathlib import Path

RUNDIR = Path(f"/glade/derecho/scratch/faezehmaghso/{CASENAME}/run")
OUTDIR = WORKSPACE / "mom6_output" / CASENAME
OUTDIR.mkdir(parents=True, exist_ok=True)


def tidy(ds):
    """Box output uses xh_sub01/yh_sub01; rename to xh/yh."""
    ren = {d: d.split("_sub")[0] for d in ds.dims if "_sub" in d}
    ren.update({c: c.split("_sub")[0] for c in ds.coords if "_sub" in c and c not in ren})
    return ds.rename(ren)


def open_mom6(path):
    raw = xr.open_dataset(path, decode_times=False)
    raw.time.attrs["calendar"] = "proleptic_gregorian"   # MOM6's real calendar
    return tidy(xr.decode_cf(raw))


def join_tiles(box):
    out = OUTDIR / f"{CASENAME}.mom6.{box}.nc"
    if not out.exists():
        files = sorted(RUNDIR.glob(f"{CASENAME}.mom6.{box}.nc.*"))
        ds = tidy(xr.open_mfdataset(files, combine="by_coords", decode_times=False))
        ds.load().to_netcdf(out)
        print(f"{box}: joined {len(files)} pieces")
    ds = open_mom6(out)
    print(f"{box}: {out.name} {out.stat().st_size/1e6:.1f} MB, {dict(ds.sizes)}, "
          f"{str(ds.time.values[0])[:16]} -> {str(ds.time.values[-1])[:16]}")
    return ds


tx = join_tiles("sfincs_TX")
ms = join_tiles("sfincs_MS")
try:
    wtx = join_tiles("wind_TX")          # taux, tauy, SSU, SSV (2-hourly)
except Exception as e:
    wtx = None
    print("wind_TX not available:", e)

hourly = OUTDIR / f"{CASENAME}.mom6.ssh_hourly.nc"
if not hourly.exists():
    shutil.copy(RUNDIR / hourly.name, hourly)
g = open_mom6(hourly)
print("ssh_hourly:", dict(g.sizes))

# Custom history names are not archived by the stock CIME rules; preserve
# the daily surface file in permanent work storage alongside hourly SSH.
daily = OUTDIR / f"{CASENAME}.mom6.h.sfc_daily.nc"
if not daily.exists():
    shutil.copy2(RUNDIR / daily.name, daily)
d = open_mom6(daily)
print(f"sfc_daily: {daily.name} {daily.stat().st_size/1e6:.1f} MB, "
      f"{str(d.time.values[0])[:16]} -> {str(d.time.values[-1])[:16]}")

### 7a. Where the model cells are, and model sea level

Left: mean sea level over the run, the gauge (red x) and the nearest **ocean** cell used (white o).
Gauges sit in land cells at 1/12 degree, so the model cell is a few km offshore.
Right: MOM6 sea level at that cell, in the model's own datum (not NAVD88 or MSL).

In [ ]:
def nearest_wet(ds, lon, lat):
    ssh = ds["SSH_inst"]
    wet = ssh.isel(time=0, drop=True).notnull()
    d2 = (((ds.xh - lon) * np.cos(np.deg2rad(lat)))**2 + (ds.yh - lat)**2)
    d2 = d2.where(wet).transpose("yh", "xh").values
    iy, ix = np.unravel_index(np.nanargmin(d2), d2.shape)
    return ssh.isel(yh=iy, xh=ix), float(np.sqrt(d2[iy, ix])) * 111


gauges = {  # name: (lon 0-360, lat, dataset)
    "Galveston Pier 21 (TX)": (360 - 94.793, 29.31, tx),
    "Galveston Bay Entrance N Jetty (TX)": (360 - 94.725, 29.357, tx),
    "Sabine Pass North (TX)": (360 - 93.87, 29.728, tx),
}
landfall = np.datetime64("2008-09-13T07:00")   # Ike landfall at Galveston about 2008-09-13 07 UTC

fig, axes = plt.subplots(len(gauges), 2, figsize=(14, 3.3 * len(gauges)),
                         gridspec_kw={"width_ratios": [1, 2]})
for row, (name, (lon, lat, ds)) in enumerate(gauges.items()):
    s, km = nearest_wet(ds, lon, lat)
    ds["SSH_inst"].mean("time").plot(ax=axes[row, 0], cmap="viridis", add_colorbar=False)
    axes[row, 0].plot(lon, lat, "rx", ms=10, label="gauge")
    axes[row, 0].plot(float(s.xh), float(s.yh), "wo", mfc="none", ms=10, label="model cell")
    axes[row, 0].set_title(name, fontsize=10)
    axes[row, 0].legend(fontsize=8)
    s.plot(ax=axes[row, 1])
    axes[row, 1].axvline(landfall, color="r", ls="--", lw=1)
    axes[row, 1].set_title(f"model cell {float(s.xh):.2f}E, {float(s.yh):.2f}N ({km:.0f} km from gauge); red = ike landfall",
                           fontsize=10)
    axes[row, 1].set_ylabel("SSH_inst (m, model datum)")
plt.tight_layout()

### 7b. Sea level at the gauges vs NOAA

Uses the model cell nearest each gauge that is ocean (gauges sit in land cells at 1/12 degree).
Both series are anomalies from their calm-period mean, so the datum doesn't matter here.
The printed NOAA station name confirms each ID.

In [ ]:
import requests


def nearest_wet(ds, lon, lat):
    ssh = ds["SSH_inst"]
    wet = ssh.isel(time=0, drop=True).notnull()
    d2 = (((ds.xh - lon) * np.cos(np.deg2rad(lat)))**2 + (ds.yh - lat)**2)
    d2 = d2.where(wet).transpose("yh", "xh").values
    iy, ix = np.unravel_index(np.nanargmin(d2), d2.shape)
    return ssh.isel(yh=iy, xh=ix), float(np.sqrt(d2[iy, ix])) * 111


def noaa_wl(station, begin, end, datum="MSL"):
    r = requests.get("https://api.tidesandcurrents.noaa.gov/api/prod/datagetter", params=dict(
        product="water_level", station=station, begin_date=begin, end_date=end,
        datum=datum, units="metric", time_zone="gmt", format="json",
        application="MOM6_SFINCS_check"), timeout=60).json()
    if "error" in r:
        print(station, "->", r["error"])
        return None, None
    df = pd.DataFrame(r["data"])
    s = pd.Series(pd.to_numeric(df["v"], errors="coerce").values, index=pd.to_datetime(df["t"]))
    return s, r["metadata"]["name"]


# Ike landfall at Galveston about 2008-09-13 07 UTC
landfall = np.datetime64("2008-09-13T07:00")
calm = slice("2008-09-01", "2008-09-07")
begin, end = "20080830", "20080919"
stations = {  # name: (NOAA id, lon 0-360, lat, dataset)
    "Galveston Pier 21 (TX)": ("8771450", 360 - 94.793, 29.31, tx),
    "Galveston Bay Entrance N Jetty (TX)": ("8771341", 360 - 94.725, 29.357, tx),
    "Sabine Pass North (TX)": ("8770570", 360 - 93.87, 29.728, tx),
}

fig, axes = plt.subplots(len(stations), 1, figsize=(13, 3.2 * len(stations)), sharex=True)
for ax, (name, (sid, lon, lat, ds)) in zip(axes, stations.items()):
    mod, km = nearest_wet(ds, lon, lat)
    mod = mod.to_series()
    ax.plot(mod.index, mod - mod[calm].mean(), label=f"MOM6 ({km:.0f} km offshore)")
    obs, noaa_name = noaa_wl(sid, begin, end)
    title = name
    if obs is not None:
        ax.plot(obs.index, obs - obs[calm].mean(), label=f"NOAA {sid} {noaa_name}", alpha=0.8)
        both = pd.concat([mod.resample("1h").mean() - mod[calm].mean(),
                          obs.resample("1h").mean() - obs[calm].mean()], axis=1).dropna()
        rmse = float(np.sqrt(((both.iloc[:, 0] - both.iloc[:, 1])**2).mean()))
        title += f": hourly RMSE {rmse:.2f} m"
    ax.axvline(landfall, color="r", ls="--", lw=1)
    ax.set_title(title, fontsize=10)
    ax.set_ylabel("anomaly (m)")
    ax.legend(fontsize=8)
plt.tight_layout()

### 7d. This experiment vs the base run (E0) and NOAA

Same gauges, same anomaly method (calm-period mean removed). The table is saved to your home folder.
A change is useful if `peak_err_m` and the forerunner move toward NOAA without making the tides (before 11 Sep) worse.

In [ ]:
from IPython.display import display

BASE_TX = WORKSPACE / "mom6_output" / "gom12_ike.001" / "gom12_ike.001.mom6.sfincs_TX.nc"
RUNS = {"E0 base": BASE_TX, CASENAME: OUTDIR / f"{CASENAME}.mom6.sfincs_TX.nc"}
pre = slice(str(landfall - np.timedelta64(24, "h")), str(landfall))   # forerunner window

rows = []
fig, axes = plt.subplots(len(stations), 1, figsize=(13, 3.2 * len(stations)), sharex=True)
for ax, (name, (sid, lon, lat, _)) in zip(axes, stations.items()):
    obs, _n = noaa_wl(sid, begin, end)
    o = (obs - obs[calm].mean()).resample("1h").mean() if obs is not None else None
    if o is not None:
        ax.plot(o.index, o, "k", lw=1.5, label="NOAA")
    for label, path in RUNS.items():
        m = nearest_wet(open_mom6(path), lon, lat)[0].to_series()
        m = (m - m[calm].mean()).resample("1h").mean()
        ax.plot(m.index, m, label=label)
        r = {"station": name, "run": label, "peak_model_m": m.max(), "peak_time": m.idxmax(),
             "forerunner_model_m": m[pre].mean()}
        if o is not None:
            both = pd.concat([m, o], axis=1, keys=["m", "o"]).dropna()
            r.update(rmse_m=np.sqrt(((both.m - both.o) ** 2).mean()), bias_m=(both.m - both.o).mean(),
                     peak_obs_m=o.max(), peak_err_m=m.max() - o.max(), forerunner_obs_m=o[pre].mean())
        rows.append(r)
    ax.axvline(landfall, color="r", ls="--", lw=1)
    ax.set_title(name, fontsize=10)
    ax.set_ylabel("anomaly (m)")
    ax.legend(fontsize=8)
plt.tight_layout()

score = pd.DataFrame(rows).round(3)
display(score)
score.to_csv(Path.home() / f"{CASENAME}_vs_base_scorecard.csv", index=False)

### 7e. Wind stress and surface current offshore of Galveston

The strength of the wind stress during 11 to 13 Sep is what builds the forerunner. Compare it between experiments (E0 has no wind output).

In [ ]:
if wtx is None:
    print("no wind_TX output in this run")
else:
    def at_point(da, lon, lat):
        xd = [d for d in da.dims if d.startswith("x")][0]
        yd = [d for d in da.dims if d.startswith("y")][0]
        return da.sel({xd: lon, yd: lat}, method="nearest")

    lon, lat = 360 - 94.6, 29.0          # open water ~30 km off Galveston
    taux, tauy = at_point(wtx["taux"], lon, lat), at_point(wtx["tauy"], lon, lat)
    ssu, ssv = at_point(wtx["SSU"], lon, lat), at_point(wtx["SSV"], lon, lat)
    fig, ax = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
    ax[0].plot(taux.time, np.hypot(taux.values, tauy.values), label="|wind stress|")
    ax[0].plot(taux.time, taux, lw=0.8, label="taux (east)")
    ax[0].plot(tauy.time, tauy, lw=0.8, label="tauy (north)")
    ax[0].set_ylabel("stress (Pa)")
    ax[1].plot(ssu.time, np.hypot(ssu.values, ssv.values), label="|surface current|")
    ax[1].set_ylabel("current (m/s)")
    for a in ax:
        a.axvline(landfall, color="r", ls="--", lw=1)
        a.legend(fontsize=8)
    ax[0].set_title(f"{CASENAME}: 94.6 W, 29.0 N", fontsize=10)
    plt.tight_layout()

### 7c. Animation of sea level (whole Gulf, hourly)

Anomaly from the run mean, 2008-09-09 to 2008-09-15.
The GIF is saved in `OUTDIR` and copied to your home folder; download it from the JupyterLab file browser (right-click, then Download).

In [ ]:
from matplotlib import animation
from IPython.display import Image

ssh = g["SSH_inst"]
anom = (ssh - ssh.mean("time")).sel(time=slice("2008-09-09", "2008-09-15")).load()

fig, ax = plt.subplots(figsize=(9, 6))
ax.set_facecolor("0.75")
im = ax.pcolormesh(anom.xh, anom.yh, np.ma.masked_invalid(anom.isel(time=0).values),
                   cmap="RdBu_r", vmin=-0.8, vmax=0.8, shading="nearest")
fig.colorbar(im, ax=ax, label="SSH anomaly from run mean (m)")
ax.plot(360 - 94.793, 29.31, "k^", ms=6)
ax.set_xlabel("longitude (deg E)")
ax.set_ylabel("latitude (deg N)")
title = ax.set_title("")


def update(i):
    im.set_array(np.ma.masked_invalid(anom.isel(time=i).values))
    title.set_text(f"MOM6 Gulf 1/12 deg, ike: sea level anomaly {str(anom.time.values[i])[:16]} UTC")
    return im, title


anim = animation.FuncAnimation(fig, update, frames=anom.sizes["time"], interval=100)
gif = OUTDIR / f"{CASENAME}_ssh_ike.gif"
anim.save(gif, writer=animation.PillowWriter(fps=10), dpi=80)
plt.close(fig)
shutil.copy(gif, Path.home() / gif.name)
print("saved", gif, "and a copy in your home folder")
Image(filename=str(gif))